### RAG Pilelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_24520/3728471183.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/home/akapandey/Projects/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Read all the pdf's inside the dir

def process_all_pdfs(pdf_directory):
    """process all PDF files in the dir"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf files recursively 
    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to procress.")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Processed {len(documents)} documents from {pdf_file.name}")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data/pdf")

Found 1 PDF files to procress.
Processing Statement Of Purpose.pdf
Processed 2 documents from Statement Of Purpose.pdf


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Statement Of Purpose', 'source': '../data/pdf/Statement Of Purpose.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Statement Of Purpose.pdf', 'file_type': 'pdf'}, page_content='Statement  Of  Purpose(SOP)   \nMachine  Learning  has  become  one  of  the  most  impactful  technologies  of  our  time,  transforming  \nthe\n \nway\n \ndata\n \nis\n \nanalyzed\n \nand\n \ndecisions\n \nare\n \nmade.\n \nAs\n \na\n \nComputer\n \nScience\n \nEngineering\n \nundergraduate\n \nwith\n \na\n \nCGPA\n \nof\n \n8.0,\n \nI\n \nhave\n \ndeveloped\n \na\n \nstrong\n \ninterest\n \nin\n \nMachine\n \nLearning,\n \nData\n \nScience,\n \nand\n \nData\n \nAnalytics\n \nthrough\n \nmy\n \nacademic\n \ncoursework,\n \nprojects,\n \nand\n \ncontinuous\n \nself-learning.\n \nI\n \nam\n \nexcited\n \nabout\n \nthe\n \nopportunity\n \nto\n \nparticipate\n \nin\n \nthe\n

### Text Splitting into Chunks

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")


    if split_docs:
        print(f"\n Example Chunk:")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)
chunks


Split 2 documents into 6 chunks.

 Example Chunk:
Content : Statement  Of  Purpose(SOP)   
Machine  Learning  has  become  one  of  the  most  impactful  technologies  of  our  time,  transforming  
the
 
way
 
data
 
is
 
analyzed
 
and
 
decisions
 
are
 
ma...
Metadata : {'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Statement Of Purpose', 'source': '../data/pdf/Statement Of Purpose.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Statement Of Purpose.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Statement Of Purpose', 'source': '../data/pdf/Statement Of Purpose.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Statement Of Purpose.pdf', 'file_type': 'pdf'}, page_content='Statement  Of  Purpose(SOP)   \nMachine  Learning  has  become  one  of  the  most  impactful  technologies  of  our  time,  transforming  \nthe\n \nway\n \ndata\n \nis\n \nanalyzed\n \nand\n \ndecisions\n \nare\n \nmade.\n \nAs\n \na\n \nComputer\n \nScience\n \nEngineering\n \nundergraduate\n \nwith\n \na\n \nCGPA\n \nof\n \n8.0,\n \nI\n \nhave\n \ndeveloped\n \na\n \nstrong\n \ninterest\n \nin\n \nMachine\n \nLearning,\n \nData\n \nScience,\n \nand\n \nData\n \nAnalytics\n \nthrough\n \nmy\n \nacademic\n \ncoursework,\n \nprojects,\n \nand\n \ncontinuous\n \nself-learning.\n \nI\n \nam\n \nexcited\n \nabout\n \nthe\n \nopportunity\n \nto\n \nparticipate\n \nin\n \nthe\n

### Embeddings & Vector DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


/home/akapandey/Projects/RAG/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13932.77it/s]


Model loaded successfully. Embedding dimension: 384


In [8]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Devices:", torch.cuda.device_count())


CUDA Available: False
CUDA Devices: 1


### Vector Store

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Statement Of Purpose', 'source': '../data/pdf/Statement Of Purpose.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Statement Of Purpose.pdf', 'file_type': 'pdf'}, page_content='Statement  Of  Purpose(SOP)   \nMachine  Learning  has  become  one  of  the  most  impactful  technologies  of  our  time,  transforming  \nthe\n \nway\n \ndata\n \nis\n \nanalyzed\n \nand\n \ndecisions\n \nare\n \nmade.\n \nAs\n \na\n \nComputer\n \nScience\n \nEngineering\n \nundergraduate\n \nwith\n \na\n \nCGPA\n \nof\n \n8.0,\n \nI\n \nhave\n \ndeveloped\n \na\n \nstrong\n \ninterest\n \nin\n \nMachine\n \nLearning,\n \nData\n \nScience,\n \nand\n \nData\n \nAnalytics\n \nthrough\n \nmy\n \nacademic\n \ncoursework,\n \nprojects,\n \nand\n \ncontinuous\n \nself-learning.\n \nI\n \nam\n \nexcited\n \nabout\n \nthe\n \nopportunity\n \nto\n \nparticipate\n \nin\n \nthe\n

In [11]:
# Convert the text to embeddings

texts = [doc.page_content for doc in chunks]
texts

# Generate the emebeddings for the chunks

embeddings = embedding_manager.generate_embeddings(texts)

#Store in the vector store

vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 6 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.05it/s]

Generated embeddings with shape: (6, 384)
Adding 6 documents to vector store...
Successfully added 6 documents to vector store
Total documents in collection: 6


### retriever Pipeline from vector store

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    print(f"Distance: {distance}")
                    print(f"Similarity: {similarity_score}")
                    
                    if True:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever

In [19]:
rag_retriever.retrieve("CGPA")

Retrieving documents for query: 'CGPA'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 173.73it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.6144962310791016
Similarity: -0.6144962310791016
Distance: 1.6259033679962158
Similarity: -0.6259033679962158
Distance: 1.75872802734375
Similarity: -0.75872802734375
Distance: 1.8990254402160645
Similarity: -0.8990254402160645
Distance: 1.9231128692626953
Similarity: -0.9231128692626953
Retrieved 5 documents (after filtering)


[{'id': 'doc_16937efc_0',
  'content': 'Statement  Of  Purpose(SOP)   \nMachine  Learning  has  become  one  of  the  most  impactful  technologies  of  our  time,  transforming  \nthe\n \nway\n \ndata\n \nis\n \nanalyzed\n \nand\n \ndecisions\n \nare\n \nmade.\n \nAs\n \na\n \nComputer\n \nScience\n \nEngineering\n \nundergraduate\n \nwith\n \na\n \nCGPA\n \nof\n \n8.0,\n \nI\n \nhave\n \ndeveloped\n \na\n \nstrong\n \ninterest\n \nin\n \nMachine\n \nLearning,\n \nData\n \nScience,\n \nand\n \nData\n \nAnalytics\n \nthrough\n \nmy\n \nacademic\n \ncoursework,\n \nprojects,\n \nand\n \ncontinuous\n \nself-learning.\n \nI\n \nam\n \nexcited\n \nabout\n \nthe\n \nopportunity\n \nto\n \nparticipate\n \nin\n \nthe\n \nAmazon\n \nML\n \nSummer\n \nSchool\n \nto\n \nstrengthen\n \nmy\n \nunderstanding\n \nof\n \nmachine\n \nlearning\n \nfundamentals\n \nand\n \nlearn\n \nfrom\n \nexperts\n \nworking\n \non\n \nreal-world\n \napplications\n \nat\n \nscale.\n \nMy  academic  curriculum  has  i

In [22]:
results = rag_retriever.retrieve(
    "several projects ",
    top_k=3,
    score_threshold=-999
)

for r in results:
    print(r["rank"])
    print(r["distance"])
    print(r["content"][:300])
    print("-"*100)

Retrieving documents for query: 'several projects '
Top K: 3, Score threshold: -999
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 163.34it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.0622411966323853
Similarity: -0.062241196632385254
Distance: 1.20310378074646
Similarity: -0.20310378074645996
Distance: 1.3467254638671875
Similarity: -0.3467254638671875
Retrieved 3 documents (after filtering)
1
1.0622411966323853
attendance
 
tracking.
 
I
 
also
 
worked
 
on
 
a
 
Pothole
 
Detection
 
System
 
using
 
RGB
 
and
 
depth
 
data,
 
where
 
I
 
explored
 
how
 
machine
 
learning
 
and
 
computer
 
vision
 
can
 
be
 
applied
 
to
 
improve
 
road
 
safety
 
and
 
infrastructure
 
monitoring.
 
Additionally,

----------------------------------------------------------------------------------------------------
2
1.20310378074646
Machine
 
Learning,
 
Data
 
Science,
 
Data
 
Analytics,
 
and
 
Data
 
Structures.
 
These
 
courses
 
have
 
helped
 
me
 
build
 
a
 
foundation
 
in
 
analytical
 
thinking,
 
algorithm
 
design,
 
and
 
data-driven
 
problem
 
solving.
 
Through
 
laboratory
 
work
 
and
 
coursework,


### Integration Vectordb Context pipeline With LLM output


In [28]:
# Simple RAG pipeline with groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=2048)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [32]:
answer = rag_simple("Where i am currently interning? and which technologies ?",rag_retriever,llm,top_k=5)
answer

Retrieving documents for query: 'Where i am currently interning? and which technologies ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 137.63it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.048368215560913
Similarity: -0.048368215560913086
Distance: 1.2659975290298462
Similarity: -0.2659975290298462
Distance: 1.3497179746627808
Similarity: -0.34971797466278076
Distance: 1.3997929096221924
Similarity: -0.3997929096221924
Distance: 1.4105805158615112
Similarity: -0.41058051586151123
Retrieved 5 documents (after filtering)


'I am currently interning at HCLTech, where I am gaining exposure to enterprise technologies and cybersecurity concepts such as Endpoint Detection and Response (EDR), File Integrity Monitoring (FIM), Privileged Access Management (PAM), and related domains.'

In [36]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what are my long-term goal?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
# print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what are my long-term goal?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 149.23it/s]

Generated embeddings with shape: (1, 384)
Distance: 1.508258581161499
Similarity: -0.508258581161499
Distance: 1.5219284296035767
Similarity: -0.5219284296035767
Distance: 1.6070449352264404
Similarity: -0.6070449352264404
Retrieved 3 documents (after filtering)


Answer: Your long-term goal is not explicitly stated in the provided context, but based on your interests and experiences, it can be inferred that your long-term goal is to work in the field of Machine Learning, Data Science, and Data Analytics, and to apply these technologies to solve complex industry challenges.
Sources: [{'source': 'Statement Of Purpose.pdf', 'page': 0, 'score': -0.508258581161499, 'preview': 'attendance\n \ntracking.\n \nI\n \nalso\n \nworked\n \non\n \na\n \nPothole\n \nDetection\n \nSystem\n \nusing\n \nRGB\n \nand\n \ndepth\n \ndata,\n \nwhere\n \nI\n \nexplored\n \nhow\n \nmachine\n \nlearning\n \nand\n \ncomputer\n \nvision\n \ncan\n \nbe\n \napplied\n \nto\n \nimprove\n \nroad\n \nsafety\n \nand\n \ninfrastructure\n \nmonitoring.\n \nAdditionally,\n...'}, {'source': 'Statement Of Purpose.pdf', 'page': 0, 'score': -0.5219284296035767, 'preview': 'Statement  Of  Purpose(SOP)   \nMachine  Learning  has  become  one  of  the  most  impactful  technologies  of  ou